# Closed-Loop RPT Validation

本 Notebook 基于 **真实 RPT summary/trace** 与 PyBaMM 仿真输出，对比 `SOH / LLI / LAM / ICA(dQ/dV)`。

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
REAL_SUMMARY = ROOT / 'data' / 'real_rpt_summary.csv'
REAL_TRACE = ROOT / 'data' / 'real_rpt_low_rate_trace.csv'
PROTOCOL_JSON = ROOT / 'configs' / 'degradation_protocol.json'
OUT_DIR = ROOT / 'models' / 'degradation' / 'validation'
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
real_summary = pd.read_csv(REAL_SUMMARY)
real_trace = pd.read_csv(REAL_TRACE)
if 'soh_pct' not in real_summary.columns:
    real_summary['soh_pct'] = 100 * real_summary['capacity_01c_ah'] / real_summary['capacity_01c_ah'].iloc[0]
real_summary.head()

In [ ]:
sim_summary_csv = OUT_DIR / 'sim_rpt_summary.csv'
sim_trace_csv = OUT_DIR / 'sim_rpt_trace.csv'
sim_feature_csv = OUT_DIR / 'sim_rpt_feature.csv'
sim_state_json = OUT_DIR / 'sim_state.json'
cmd = [
    'python', 'src/degradation/coupled_degradation_model.py',
    '--protocol-json', str(PROTOCOL_JSON),
    '--output-csv', str(sim_summary_csv),
    '--output-trace-csv', str(sim_trace_csv),
    '--output-feature-csv', str(sim_feature_csv),
    '--output-state-json', str(sim_state_json),
]
subprocess.run(cmd, cwd=ROOT, check=True)
sim_summary = pd.read_csv(sim_summary_csv)
sim_trace = pd.read_csv(sim_trace_csv)
sim_feature = pd.read_csv(sim_feature_csv)
sim_summary.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(real_summary['cycle'], real_summary['soh_pct'], 'o--', label='Real SOH')
axes[0].plot(sim_summary['cycle'], sim_summary['soh_pct'], 's-', label='Sim SOH')
axes[0].set_title('SOH')
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].plot(real_summary['cycle'], real_summary['lli_pct'], 'o--', label='Real LLI')
axes[1].plot(sim_summary['cycle'], sim_summary['lli_pct'], 's-', label='Sim LLI')
axes[1].set_title('LLI')
axes[1].grid(alpha=0.3)
axes[1].legend()
axes[2].plot(real_summary['cycle'], real_summary['lam_ne_pct'], 'o--', label='Real LAM-NE')
axes[2].plot(sim_summary['cycle'], sim_summary['lam_ne_pct'], 's-', label='Sim LAM-NE')
axes[2].plot(real_summary['cycle'], real_summary['lam_pe_pct'], 'o--', label='Real LAM-PE')
axes[2].plot(sim_summary['cycle'], sim_summary['lam_pe_pct'], 's-', label='Sim LAM-PE')
axes[2].set_title('LAM')
axes[2].grid(alpha=0.3)
axes[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
checkpoint = int(real_trace['checkpoint'].min())
role = 'low_rate_discharge'
real_curve = real_trace[(real_trace['checkpoint'] == checkpoint) & (real_trace['trace_role'] == role)]
sim_curve = sim_feature[(sim_feature['checkpoint'] == checkpoint) & (sim_feature['trace_role'] == role)]
plt.figure(figsize=(7, 4))
plt.plot(real_curve['voltage_v'], real_curve['capacity_ah'], 'o-', label='Real Q-V trace')
plt.xlabel('Voltage [V]')
plt.ylabel('Capacity [Ah]')
plt.title(f'Real low-rate trace checkpoint {checkpoint}')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(sim_curve['voltage_v'], sim_curve['ica'], '-', label='Sim ICA(dQ/dV)')
plt.xlabel('Voltage [V]')
plt.ylabel('dQ/dV')
plt.title(f'Simulated ICA checkpoint {checkpoint}')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()